# Earnings Tick Visualization - Best Strategy Selection

This notebook automatically selects the best calendar spread opportunity using liquidity analysis,
then visualizes tick-level price movements around earnings.

**Workflow:**
1. Run liquidity analysis to find best opportunities for earnings date
2. Select top-ranked contract (highest liquidity + IV ratio)
3. Load tick data for both legs (short and long)
4. Plot bid/ask spreads and price evolution
5. Calculate realistic P&L using tick data

**Prerequisites:**
- Earnings calendar data loaded
- Option chain snapshots available
- Option bars data for liquidity calculation
- Tick data backfilled for the contracts

## 2. Load IV Ratio Rankings

Use pre-computed IV ratio rankings (much faster than full liquidity analysis).

# Instead of slow liquidity analysis, use IV ratio ranking results
# This is much faster and already filters for calendar spreads

# Try to load pre-computed IV ranking results
results_csv = project_root / "notebooks" / "iv_ratio_ranking_results.csv"

if results_csv.exists():
    rprint(f"\n[bold green]Loading IV ranking results from {results_csv.name}[/bold green]")
    opportunities_df = pd.read_csv(results_csv)
    
    # Filter by earnings date if date column exists
    if 'earnings_date' in opportunities_df.columns:
        opportunities_df['earnings_date'] = pd.to_datetime(opportunities_df['earnings_date']).dt.date
        opportunities_df = opportunities_df[opportunities_df['earnings_date'] == EARNINGS_DATE]
    
    # Filter by earnings timing if column exists
    if 'earnings_timing' in opportunities_df.columns and EARNINGS_TIMING:
        opportunities_df = opportunities_df[opportunities_df['earnings_timing'] == EARNINGS_TIMING]
    
    # Filter by symbols if specified
    if SYMBOL_FILTER:
        opportunities_df = opportunities_df[opportunities_df['symbol'].isin(SYMBOL_FILTER)]
    
    if opportunities_df.empty:
        rprint(f"[bold red]No opportunities found for {EARNINGS_DATE} ({EARNINGS_TIMING})[/bold red]")
        raise ValueError("No matching opportunities")
    
    # Display results
    rprint(f"\n[bold green]Found {len(opportunities_df)} calendar spread opportunities[/bold green]")
    
    # Show top 10
    table = Table(title=f"Top IV Ratio Opportunities - {EARNINGS_DATE}")
    table.add_column("Rank", justify="right")
    table.add_column("Symbol", style="cyan bold")
    table.add_column("Strike", justify="right")
    table.add_column("Right")
    table.add_column("Short Exp")
    table.add_column("Long Exp")
    table.add_column("IV Ratio", justify="right", style="green")
    table.add_column("Entry Cost", justify="right")
    table.add_column("Timing")
    
    for i, row in opportunities_df.head(10).iterrows():
        table.add_row(
            str(i+1),
            row['symbol'],
            f"${row['strike']:.1f}",
            row.get('right', 'C'),
            str(row['short_expiry'])[:10],
            str(row['long_expiry'])[:10],
            f"{row['iv_ratio']:.3f}",
            f"${row.get('entry_cost', 0):.0f}",
            row.get('earnings_timing', EARNINGS_TIMING)
        )
    
    console.print(table)
    
else:
    rprint(f"[bold red]IV ranking results not found: {results_csv}[/bold red]")
    rprint(\"[bold yellow]Please run notebook 06c first to generate IV rankings[/bold yellow]")
    rprint(\"[bold yellow]Or provide a manual list of opportunities[/bold yellow]\")\n    \n    # Fallback to manual specification\n    rprint(\"\\n[bold cyan]Using manual fallback opportunities:[/bold cyan]\")\n    opportunities_df = pd.DataFrame([\n        {'symbol': 'ARMK', 'strike': 38.0, 'short_expiry': '2025-11-21', 'long_expiry': '2025-12-19',\n         'right': 'C', 'iv_ratio': 2.29, 'earnings_timing': 'PRE_MARKET'},\n        {'symbol': 'SOHU', 'strike': 15.0, 'short_expiry': '2025-11-21', 'long_expiry': '2025-12-19',\n         'right': 'C', 'iv_ratio': 2.26, 'earnings_timing': 'PRE_MARKET'},\n        {'symbol': 'YSG', 'strike': 7.5, 'short_expiry': '2025-11-21', 'long_expiry': '2026-01-16',\n         'right': 'C', 'iv_ratio': 2.01, 'earnings_timing': 'PRE_MARKET'},\n    ])\n    \n    # Convert dates\n    opportunities_df['short_expiry'] = pd.to_datetime(opportunities_df['short_expiry']).dt.date\n    opportunities_df['long_expiry'] = pd.to_datetime(opportunities_df['long_expiry']).dt.date

In [ ]:
# Select the top-ranked opportunity
best = opportunities_df.iloc[0]

symbol = best['symbol']
strike = best['strike']
right = best.get('right', 'C')
short_expiry = best['short_expiry'] if isinstance(best['short_expiry'], date) else pd.to_datetime(best['short_expiry']).date()
long_expiry = best['long_expiry'] if isinstance(best['long_expiry'], date) else pd.to_datetime(best['long_expiry']).date()

rprint(f"\n[bold green]Selected Best Calendar Spread:[/bold green]")
rprint(f"  Rank: #1 (Highest IV Ratio)")
rprint(f"  Symbol: {symbol}")
rprint(f"  Strike: ${strike:.1f}")
rprint(f"  Right: {right}")
rprint(f"  IV Ratio: {best['iv_ratio']:.3f}")
if 'entry_cost' in best:
    rprint(f"  Entry Cost: ${best['entry_cost']:.2f}")

rprint(f"\n[bold cyan]Short Leg (Sell):[/bold cyan]")
rprint(f"  Expiry: {short_expiry}")
rprint(f"  Contract: {symbol} ${strike}{right} exp {short_expiry}")

rprint(f"\n[bold cyan]Long Leg (Buy):[/bold cyan]")
rprint(f"  Expiry: {long_expiry}")
rprint(f"  Contract: {symbol} ${strike}{right} exp {long_expiry}")

# Calculate entry/exit windows
calculator = EarningsTimingCalculator()
windows = calculator.calculate_windows(
    earnings_date=EARNINGS_DATE,
    earnings_time=EARNINGS_TIMING,
)

rprint(f"\n[bold cyan]Trading Windows:[/bold cyan]")
rprint(f"  Entry: {windows.entry.start} to {windows.entry.end}")
rprint(f"  Exit: {windows.exit.start} to {windows.exit.end}")

# Initialize tick reader
# Note: Dataset name is "option_ticks" not "ticks"
tick_reader = OptionTicksReader(str(data_dir), "option_ticks")

# Load tick data for short leg
rprint(f"\n[bold cyan]Loading tick data for short leg...[/bold cyan]")
short_entry_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=short_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.entry.start,
    end_datetime=windows.entry.end,
    tick_type='bid_ask'
)

short_exit_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=short_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.exit.start,
    end_datetime=windows.exit.end,
    tick_type='bid_ask'
)

rprint(f"  Entry: {len(short_entry_ticks)} ticks")
rprint(f"  Exit: {len(short_exit_ticks)} ticks")

# Load tick data for long leg
rprint(f"\n[bold cyan]Loading tick data for long leg...[/bold cyan]")
long_entry_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=long_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.entry.start,
    end_datetime=windows.entry.end,
    tick_type='bid_ask'
)

long_exit_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=long_expiry,
    strike=strike,
    right=right,
    start_datetime=windows.exit.start,
    end_datetime=windows.exit.end,
    tick_type='bid_ask'
)

rprint(f"  Entry: {len(long_entry_ticks)} ticks")
rprint(f"  Exit: {len(long_exit_ticks)} ticks")

# Combine entry and exit for each leg
short_leg_ticks = pd.concat([short_entry_ticks, short_exit_ticks], ignore_index=True)
if not short_leg_ticks.empty:
    short_leg_ticks = short_leg_ticks.sort_values('tick_time')

long_leg_ticks = pd.concat([long_entry_ticks, long_exit_ticks], ignore_index=True)
if not long_leg_ticks.empty:
    long_leg_ticks = long_leg_ticks.sort_values('tick_time')

rprint(f"\n[bold green]Total ticks loaded:[/bold green]")
rprint(f"  Short leg: {len(short_leg_ticks)} ticks")
rprint(f"  Long leg: {len(long_leg_ticks)} ticks")

if short_leg_ticks.empty or long_leg_ticks.empty:
    rprint("\n[bold red]No tick data available![/bold red]")
    rprint("[bold yellow]Make sure to run:[/bold yellow]")
    rprint(f"  dlt-ibapi backfill-batch-calendar-ticks {EARNINGS_DATE.strftime('%Y-%m-%d')} --earnings-timing {EARNINGS_TIMING}")

In [3]:
# Run liquidity analysis using the same code as CLI
params = LiquidityAnalysisParams(
    database_path=data_dir,
    earnings_date=EARNINGS_DATE,
    symbols=SYMBOL_FILTER,
    min_score=30,  # Lower threshold to get more results
    top_n=10,
    earnings_dataset="earnings",
    option_chains_dataset="option_chains",
    options_dataset="options",
)

rprint("\n[bold cyan]Running liquidity analysis...[/bold cyan]")
result = execute_liquidity_analysis(params)

if not result.success:
    rprint(f"[bold red]Liquidity analysis failed: {result.error}[/bold red]")
    raise ValueError(result.error)

rprint(f"\n[bold green]Found {result.liquid_contracts_count} liquid contracts[/bold green]")
rprint(f"  Total evaluated: {result.total_contracts_evaluated}")
rprint(f"  Duration: {result.duration_seconds:.2f}s")

# Display top opportunities
if result.metrics:
    table = Table(title=f"Top Opportunities for {EARNINGS_DATE}")
    table.add_column("Rank", justify="right")
    table.add_column("Symbol", style="cyan bold")
    table.add_column("Strike", justify="right")
    table.add_column("Right")
    table.add_column("Expiry")
    table.add_column("Liquidity Score", justify="right", style="green")
    table.add_column("Avg Volume", justify="right")
    table.add_column("Avg OI", justify="right")
    table.add_column("Spread %", justify="right", style="yellow")
    
    for i, metric in enumerate(result.metrics[:10], 1):
        table.add_row(
            str(i),
            metric.symbol,
            f"${metric.strike:.1f}",
            metric.right,
            str(metric.expiry),
            f"{metric.liquidity_score:.1f}",
            f"{int(metric.avg_volume)}",
            f"{int(metric.avg_open_interest)}",
            f"{metric.avg_spread_pct:.2f}%"
        )
    
    console.print(table)
else:
    rprint("[bold red]No liquid contracts found![/bold red]")
    raise ValueError("No opportunities to analyze")

Running liquidity analysis...

2025-11-17 23:03:38 [info     ] liquidity_analysis_start       earnings_date=2025-11-17 min_score=30.0 symbols=None
2025-11-17 23:03:38 [info     ] earnings_symbols_loaded        count=43
2025-11-17 23:03:41 [info     ] contracts_loaded               count=4854


KeyboardInterrupt: 

## 3. Select Best Strategy

Pick the top-ranked contract and determine the calendar spread legs.

In [4]:
# Select top contract
best = result.metrics[0]

# For calendar spread, we need 2 expirations
# The liquidity analysis shows individual contracts, so we need to find matching strike/right with different expiry
symbol = best.symbol
strike = best.strike
right = best.right

# Find all contracts for this symbol/strike/right
matching = [m for m in result.metrics if m.symbol == symbol and m.strike == strike and m.right == right]

if len(matching) < 2:
    rprint(f"[bold yellow]Only 1 expiration found for {symbol} ${strike}{right}[/bold yellow]")
    rprint(f"[bold yellow]Using first 2 available expirations for this symbol/strike[/bold yellow]")
    matching = [m for m in result.metrics if m.symbol == symbol and m.strike == strike][:2]

if len(matching) < 2:
    rprint(f"[bold red]Cannot create calendar spread - need 2 expirations[/bold red]")
    raise ValueError("Insufficient expirations for calendar spread")

# Sort by expiry (earliest first)
matching.sort(key=lambda x: x.expiry)

short_leg = matching[0]  # Nearest expiry (sell)
long_leg = matching[1]   # Further expiry (buy)

rprint(f"\n[bold green]Selected Calendar Spread:[/bold green]")
rprint(f"  Symbol: {symbol}")
rprint(f"  Strike: ${strike:.1f}")
rprint(f"  Right: {right}")
rprint(f"\n[bold cyan]Short Leg (Sell):[/bold cyan]")
rprint(f"  Expiry: {short_leg.expiry}")
rprint(f"  Liquidity Score: {short_leg.liquidity_score:.1f}")
rprint(f"  Avg Volume: {int(short_leg.avg_volume)}")
rprint(f"  Avg Spread: ${short_leg.avg_spread:.4f} ({short_leg.avg_spread_pct:.2f}%)")
rprint(f"\n[bold cyan]Long Leg (Buy):[/bold cyan]")
rprint(f"  Expiry: {long_leg.expiry}")
rprint(f"  Liquidity Score: {long_leg.liquidity_score:.1f}")
rprint(f"  Avg Volume: {int(long_leg.avg_volume)}")
rprint(f"  Avg Spread: ${long_leg.avg_spread:.4f} ({long_leg.avg_spread_pct:.2f}%)")

# Calculate entry/exit windows
calculator = EarningsTimingCalculator()
windows = calculator.calculate_windows(
    earnings_date=EARNINGS_DATE,
    earnings_time=EARNINGS_TIMING,
)

rprint(f"\n[bold cyan]Trading Windows:[/bold cyan]")
rprint(f"  Entry: {windows.entry.start} to {windows.entry.end}")
rprint(f"  Exit: {windows.exit.start} to {windows.exit.end}")

NameError: name 'result' is not defined

rprint("\n[bold cyan]ANALYSIS SUMMARY:[/bold cyan]")
rprint(f"\n[bold green]Selected Strategy (Rank #1):[/bold green]")
rprint(f"  Symbol: {symbol}")
rprint(f"  Strike: ${strike:.1f}{right}")
rprint(f"  Short Expiry: {short_expiry} (Sell)")
rprint(f"  Long Expiry: {long_expiry} (Buy)")
rprint(f"  IV Ratio: {best['iv_ratio']:.3f}")
if 'entry_cost' in best:
    rprint(f"  Entry Cost: ${best['entry_cost']:.2f}")

rprint("\n[bold cyan]KEY INSIGHTS:[/bold cyan]")
rprint(f"  • Top-ranked by IV ratio ({best['iv_ratio']:.3f}x)")
rprint("  • Strategy: Sell near-term (high IV) + Buy longer-term (lower IV)")
rprint("  • Profit from IV crush on short leg after earnings")
rprint("  • Tick data shows realistic execution costs from bid/ask spreads")
rprint(f"  • Earnings: {EARNINGS_DATE} {EARNINGS_TIMING}")

if not short_leg_ticks.empty and not long_leg_ticks.empty:
    rprint("\n[bold green]✓ Tick data successfully loaded for visualization[/bold green]")
else:
    rprint("\n[bold yellow]⚠ No tick data - run backfill command to see visualizations[/bold yellow]")

In [ ]:
# Initialize tick reader
tick_reader = OptionTicksReader(str(data_dir), "ticks")

# Load tick data for short leg
rprint(f"\n[bold cyan]Loading tick data for short leg...[/bold cyan]")
short_entry_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=short_leg.expiry,
    strike=strike,
    right=right,
    start_datetime=windows.entry.start,
    end_datetime=windows.entry.end,
    tick_type='bid_ask'
)

short_exit_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=short_leg.expiry,
    strike=strike,
    right=right,
    start_datetime=windows.exit.start,
    end_datetime=windows.exit.end,
    tick_type='bid_ask'
)

rprint(f"  Entry: {len(short_entry_ticks)} ticks")
rprint(f"  Exit: {len(short_exit_ticks)} ticks")

# Load tick data for long leg
rprint(f"\n[bold cyan]Loading tick data for long leg...[/bold cyan]")
long_entry_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=long_leg.expiry,
    strike=strike,
    right=right,
    start_datetime=windows.entry.start,
    end_datetime=windows.entry.end,
    tick_type='bid_ask'
)

long_exit_ticks = tick_reader.get_ticks(
    underlying=symbol,
    expiry=long_leg.expiry,
    strike=strike,
    right=right,
    start_datetime=windows.exit.start,
    end_datetime=windows.exit.end,
    tick_type='bid_ask'
)

rprint(f"  Entry: {len(long_entry_ticks)} ticks")
rprint(f"  Exit: {len(long_exit_ticks)} ticks")

# Combine entry and exit for each leg
short_leg_ticks = pd.concat([short_entry_ticks, short_exit_ticks], ignore_index=True)
if not short_leg_ticks.empty:
    short_leg_ticks = short_leg_ticks.sort_values('tick_time')

long_leg_ticks = pd.concat([long_entry_ticks, long_exit_ticks], ignore_index=True)
if not long_leg_ticks.empty:
    long_leg_ticks = long_leg_ticks.sort_values('tick_time')

rprint(f"\n[bold green]Total ticks loaded:[/bold green]")
rprint(f"  Short leg: {len(short_leg_ticks)} ticks")
rprint(f"  Long leg: {len(long_leg_ticks)} ticks")

if short_leg_ticks.empty or long_leg_ticks.empty:
    rprint("\n[bold red]No tick data available![/bold red]")
    rprint("[bold yellow]Make sure to run:[/bold yellow]")
    rprint(f"  dlt-ibapi backfill-batch-calendar-ticks {EARNINGS_DATE.strftime('%Y-%m-%d')} --earnings-timing {EARNINGS_TIMING}")

## 5. Plot Short Leg (Near-Term Expiration)

Visualize bid/ask spread and midpoint evolution for the short leg.

In [ ]:
def plot_option_ticks(ticks_df, leg_name, expiry, windows, earnings_date, earnings_timing, symbol, strike, right):
    """
    Plot bid/ask tick data with entry/exit windows highlighted.
    """
    if ticks_df.empty:
        rprint(f"[bold red]No tick data to plot for {leg_name} leg[/bold red]")
        return
    
    # Calculate midpoint
    ticks_df['midpoint'] = (ticks_df['bid_price'] + ticks_df['ask_price']) / 2
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    
    # Convert tick_time to datetime if needed
    if not pd.api.types.is_datetime64_any_dtype(ticks_df['tick_time']):
        ticks_df['tick_time'] = pd.to_datetime(ticks_df['tick_time'])
    
    # Plot 1: Bid/Ask/Midpoint
    ax1.plot(ticks_df['tick_time'], ticks_df['bid_price'], 'b-', alpha=0.6, linewidth=1, label='Bid')
    ax1.plot(ticks_df['tick_time'], ticks_df['ask_price'], 'r-', alpha=0.6, linewidth=1, label='Ask')
    ax1.plot(ticks_df['tick_time'], ticks_df['midpoint'], 'g-', linewidth=2, label='Midpoint')
    ax1.fill_between(ticks_df['tick_time'], ticks_df['bid_price'], ticks_df['ask_price'], alpha=0.2, color='gray')
    
    # Highlight entry window
    entry_start = pd.to_datetime(windows.entry.start)
    entry_end = pd.to_datetime(windows.entry.end)
    ax1.axvspan(entry_start, entry_end, alpha=0.2, color='blue', label='Entry Window')
    
    # Highlight exit window
    exit_start = pd.to_datetime(windows.exit.start)
    exit_end = pd.to_datetime(windows.exit.end)
    ax1.axvspan(exit_start, exit_end, alpha=0.2, color='orange', label='Exit Window')
    
    # Mark earnings announcement time
    if earnings_timing == 'PRE_MARKET':
        earnings_time = pd.Timestamp(earnings_date) + pd.Timedelta(hours=8)  # 8am
        label = f'Earnings (Pre-Market {earnings_date})'
    else:
        earnings_time = pd.Timestamp(earnings_date) + pd.Timedelta(hours=16)  # 4pm
        label = f'Earnings (After-Hours {earnings_date})'
    
    ax1.axvline(earnings_time, color='red', linestyle='--', linewidth=2, label=label)
    
    ax1.set_ylabel('Price ($)', fontsize=12, fontweight='bold')
    ax1.set_title(f'{leg_name} Leg - {symbol} ${strike}{right} (Exp: {expiry})', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Spread percentage
    ax2.plot(ticks_df['tick_time'], ticks_df['spread_pct'], 'purple', linewidth=1.5, label='Spread %')
    ax2.axvspan(entry_start, entry_end, alpha=0.2, color='blue')
    ax2.axvspan(exit_start, exit_end, alpha=0.2, color='orange')
    ax2.axvline(earnings_time, color='red', linestyle='--', linewidth=2)
    
    ax2.set_xlabel('Time', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Spread %', fontsize=12, fontweight='bold')
    ax2.set_title(f'{leg_name} Leg - Bid/Ask Spread Percentage', fontsize=14, fontweight='bold')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # Format x-axis
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    rprint(f"\n[bold cyan]{leg_name} Leg Statistics:[/bold cyan]")
    rprint(f"  Avg Bid: ${ticks_df['bid_price'].mean():.4f}")
    rprint(f"  Avg Ask: ${ticks_df['ask_price'].mean():.4f}")
    rprint(f"  Avg Midpoint: ${ticks_df['midpoint'].mean():.4f}")
    rprint(f"  Avg Spread: ${ticks_df['spread'].mean():.4f}")
    rprint(f"  Avg Spread %: {ticks_df['spread_pct'].mean():.2f}%")
    rprint(f"  Price Range: ${ticks_df['midpoint'].min():.4f} - ${ticks_df['midpoint'].max():.4f}")
    
    # Entry vs Exit comparison
    entry_ticks = ticks_df[(ticks_df['tick_time'] >= entry_start) & (ticks_df['tick_time'] <= entry_end)]
    exit_ticks = ticks_df[(ticks_df['tick_time'] >= exit_start) & (ticks_df['tick_time'] <= exit_end)]
    
    if not entry_ticks.empty and not exit_ticks.empty:
        entry_mid = entry_ticks['midpoint'].mean()
        exit_mid = exit_ticks['midpoint'].mean()
        change = exit_mid - entry_mid
        change_pct = (change / entry_mid) * 100
        
        rprint(f"\n[bold green]Entry → Exit Change:[/bold green]")
        rprint(f"  Entry Avg: ${entry_mid:.4f}")
        rprint(f"  Exit Avg: ${exit_mid:.4f}")
        rprint(f"  Change: ${change:.4f} ({change_pct:+.2f}%)")

# Plot short leg
plot_option_ticks(
    ticks_df=short_leg_ticks,
    leg_name='Short',
    expiry=short_leg.expiry,
    windows=windows,
    earnings_date=EARNINGS_DATE,
    earnings_timing=EARNINGS_TIMING,
    symbol=symbol,
    strike=strike,
    right=right
)

## 6. Plot Long Leg (Longer-Term Expiration)

Visualize bid/ask spread and midpoint evolution for the long leg.

In [ ]:
# Plot long leg
plot_option_ticks(
    ticks_df=long_leg_ticks,
    leg_name='Long',
    expiry=long_leg.expiry,
    windows=windows,
    earnings_date=EARNINGS_DATE,
    earnings_timing=EARNINGS_TIMING,
    symbol=symbol,
    strike=strike,
    right=right
)

## 7. Calendar Spread Net Value

Plot the net value of the calendar spread (Long - Short) over time.

In [ ]:
if not short_leg_ticks.empty and not long_leg_ticks.empty:
    # Merge on tick_time to align data
    short_leg_ticks = short_leg_ticks.sort_values('tick_time').reset_index(drop=True)
    long_leg_ticks = long_leg_ticks.sort_values('tick_time').reset_index(drop=True)
    
    # Calculate midpoint for both
    short_leg_ticks['midpoint'] = (short_leg_ticks['bid_price'] + short_leg_ticks['ask_price']) / 2
    long_leg_ticks['midpoint'] = (long_leg_ticks['bid_price'] + long_leg_ticks['ask_price']) / 2
    
    # Merge using asof (forward fill)
    merged = pd.merge_asof(
        long_leg_ticks[['tick_time', 'midpoint']].rename(columns={'midpoint': 'long_mid'}),
        short_leg_ticks[['tick_time', 'midpoint']].rename(columns={'midpoint': 'short_mid'}),
        on='tick_time',
        direction='nearest',
        tolerance=pd.Timedelta('30s')
    )
    
    # Calculate spread net value (Long - Short)
    merged['spread_value'] = merged['long_mid'] - merged['short_mid']
    merged = merged.dropna()
    
    if not merged.empty:
        # Plot
        fig, ax = plt.subplots(figsize=(16, 6))
        
        ax.plot(merged['tick_time'], merged['spread_value'], 'darkgreen', linewidth=2, label='Calendar Spread Value (Long - Short)')
        ax.fill_between(merged['tick_time'], 0, merged['spread_value'], alpha=0.3, color='green')
        
        # Highlight windows
        entry_start = pd.to_datetime(windows.entry.start)
        entry_end = pd.to_datetime(windows.entry.end)
        exit_start = pd.to_datetime(windows.exit.start)
        exit_end = pd.to_datetime(windows.exit.end)
        
        ax.axvspan(entry_start, entry_end, alpha=0.2, color='blue', label='Entry Window')
        ax.axvspan(exit_start, exit_end, alpha=0.2, color='orange', label='Exit Window')
        
        # Mark earnings
        if EARNINGS_TIMING == 'PRE_MARKET':
            earnings_time = pd.Timestamp(EARNINGS_DATE) + pd.Timedelta(hours=8)
            label = f'Earnings (Pre-Market {EARNINGS_DATE})'
        else:
            earnings_time = pd.Timestamp(EARNINGS_DATE) + pd.Timedelta(hours=16)
            label = f'Earnings (After-Hours {EARNINGS_DATE})'
        
        ax.axvline(earnings_time, color='red', linestyle='--', linewidth=2, label=label)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
        
        ax.set_xlabel('Time', fontsize=12, fontweight='bold')
        ax.set_ylabel('Spread Value ($)', fontsize=12, fontweight='bold')
        ax.set_title(f'{symbol} Calendar Spread Net Value (${strike}{right})', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        # Format x-axis
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
        plt.xticks(rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        entry_spread = merged[(merged['tick_time'] >= entry_start) & (merged['tick_time'] <= entry_end)]['spread_value'].mean()
        exit_spread = merged[(merged['tick_time'] >= exit_start) & (merged['tick_time'] <= exit_end)]['spread_value'].mean()
        
        rprint(f"\n[bold cyan]Calendar Spread Statistics:[/bold cyan]")
        rprint(f"  Entry Avg Spread: ${entry_spread:.4f}")
        rprint(f"  Exit Avg Spread: ${exit_spread:.4f}")
        rprint(f"  Change: ${exit_spread - entry_spread:.4f}")
        rprint(f"  P&L per contract: ${(exit_spread - entry_spread) * 100:.2f}")
    else:
        rprint("[bold red]No overlapping tick data for calendar spread calculation[/bold red]")
else:
    rprint("[bold red]Insufficient tick data to plot calendar spread[/bold red]")

## 8. Summary

Key insights from the analysis.

In [ ]:
rprint("\n[bold cyan]ANALYSIS SUMMARY:[/bold cyan]")
rprint(f"\n[bold green]Selected Strategy:[/bold green]")
rprint(f"  Symbol: {symbol}")
rprint(f"  Strike: ${strike:.1f}{right}")
rprint(f"  Short Expiry: {short_leg.expiry} (Sell)")
rprint(f"  Long Expiry: {long_leg.expiry} (Buy)")

rprint(f"\n[bold green]Liquidity Metrics:[/bold green]")
rprint(f"  Short Leg Score: {short_leg.liquidity_score:.1f} ({short_leg.liquidity_quartile})")
rprint(f"  Long Leg Score: {long_leg.liquidity_score:.1f} ({long_leg.liquidity_quartile})")
rprint(f"  Short Leg Spread: {short_leg.avg_spread_pct:.2f}%")
rprint(f"  Long Leg Spread: {long_leg.avg_spread_pct:.2f}%")

rprint("\n[bold cyan]KEY INSIGHTS:[/bold cyan]")
rprint("  • This is the highest-ranked liquid opportunity for this earnings date")
rprint("  • Strategy: Sell near-term (high IV) + Buy longer-term (lower IV)")
rprint("  • Profit from IV crush on short leg after earnings")
rprint("  • Tick data shows realistic execution costs from bid/ask spreads")